In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import os
from pathlib import Path

BASE_EVAL_DIR = Path("/content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/eval")
BASELINE_DIR = BASE_EVAL_DIR / "baseline_evaluations"

BASELINE_DIR.mkdir(parents=True, exist_ok=True)

print("Base eval dir:", BASE_EVAL_DIR)
print("Baseline dir:", BASELINE_DIR)

Base eval dir: /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/eval
Baseline dir: /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/eval/baseline_evaluations


In [3]:
!pip -q install -U lm-eval
!pip -q install -U transformers accelerate datasets sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 kB 5.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 79.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 156.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.8/526.8 kB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 52.5 MB/s eta 0:00:00


In [ ]:
!lm-eval ls tasks | grep -E "squadv2|coqa|hotpotqa"

|coqa                                                                                  |lm_eval/tasks/coqa/default.yaml                                                                                                                                 |generate_until       |
|longbench_hotpotqa                                                                    |lm_eval/tasks/longbench/hotpotqa.yaml                                                                                                                           |                     |
|longbench_hotpotqa_e                                                                  |lm_eval/tasks/longbench/hotpotqa_e.yaml                                                                                                                         |                     |
|squadv2                                                                               |lm_eval/tasks/squadv2/squadv2.yaml                                                              

In [8]:
MODELS = [
    #{"label": "Qwen2-0.5B", "hf": "Qwen/Qwen2-0.5B", "chat": False},
    {"label": "Qwen2-1.5B-Instruct", "hf": "Qwen/Qwen2-1.5B-Instruct", "chat": True},
    #{"label": "OPT-125M", "hf": "facebook/opt-125m", "chat": False},
    #{"label": "OPT-1.3B", "hf": "facebook/opt-1.3b", "chat": False},
]
'''
TASKS = [
    "squadv2",
    "coqa",
    "hotpotqa",
]
'''
TASKS = [
    "squadv2"
]

In [11]:
import subprocess
import time
import torch
import json
import shutil
from pathlib import Path

def run_lm_eval(
    model_label,
    model_name,
    task_name,
    chat=False,
    batch_size=8,
    limit=None,
    max_gen_toks=32,
):
    model_dir = BASELINE_DIR / model_label / task_name
    model_dir.mkdir(parents=True, exist_ok=True)

    # ===== LOG START =====
    print("\n" + "="*80)
    print("[START] Evaluation")
    print(f"Model: {model_label} ({model_name})")
    print(f"Task: {task_name}")
    print(f"Batch size: {batch_size}")
    print(f"max_gen_toks: {max_gen_toks}")
    print(f"Limit: {limit}")
    print(f"Output dir: {model_dir}")

    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    else:
        print("GPU: NOT AVAILABLE ⚠️")

    print("="*80 + "\n")

    model_args = [
        f"pretrained={model_name}",
        "dtype=float16",
        "trust_remote_code=True",
    ]

    gen_kwargs = f"max_gen_toks={max_gen_toks},max_new_tokens={max_gen_toks}"

    cmd = [
        "lm-eval", "run",
        "--model", "hf",
        "--model_args", ",".join(model_args),
        "--tasks", task_name,
        "--device", "cuda:0",
        "--batch_size", str(batch_size),
        "--output_path", str(model_dir),
        "--gen_kwargs", gen_kwargs,
        "--log_samples"
    ]

    if chat:
        cmd.append("--apply_chat_template")

    if limit is not None:
        cmd += ["--limit", str(limit)]

    print("[COMMAND]")
    print(" ".join(cmd))
    print("\n[INFO] Launching lm-eval...\n")

    start = time.time()

    # ===== LIVE LOGGING =====
    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )

    first_output = True

    for line in process.stdout:
        if first_output:
            print("[INFO] lm-eval started producing output...\n")
            first_output = False
        print(line, end="")

    process.wait()
    elapsed = time.time() - start

    if process.returncode != 0:
        print("\n[ERROR] lm-eval failed ❌")
        raise subprocess.CalledProcessError(process.returncode, cmd)

    # ===== POST PROCESS =====
    print("\n[POST] Cleaning results...")

    json_files = list(model_dir.rglob("*.json"))

    result_file = None
    for f in json_files:
        if "samples" not in f.name:
            try:
                data = json.loads(f.read_text())
                if "results" in data:
                    result_file = f
                    break
            except:
                pass

    if result_file is None:
        print("[WARN] No results file found!")
        return model_dir

    data = json.loads(result_file.read_text())
    task_results = data["results"][task_name]

    clean_path = model_dir / "results_FullDatasetWithSampleDebug.json"
    with open(clean_path, "w") as f:
        json.dump(task_results, f, indent=2)

    print(f"[SAVED] Clean results → {clean_path}")

    # remove nested dirs
    for subdir in model_dir.iterdir():
        if subdir.is_dir():
            shutil.rmtree(subdir)

    print("[CLEANED] Removed nested folders")

    # ===== LOG END =====
    print("\n" + "="*80)
    print(f"[DONE] {model_label} | {task_name}")
    print(f"Time: {elapsed/60:.2f} minutes")
    print(f"Final file: {clean_path}")
    print("="*80 + "\n")

    return model_dir

In [13]:
import subprocess
import time
import torch
import json
import shutil
from pathlib import Path

def run_lm_eval(
    model_label,
    model_name,
    task_name,
    chat=False,
    batch_size=8,
    limit=None,
    max_gen_toks=32,
    copy_samples_to_root=True,
):
    model_dir = BASELINE_DIR / model_label / task_name
    model_dir.mkdir(parents=True, exist_ok=True)

    print("\n" + "=" * 80)
    print("[START] Evaluation")
    print(f"Model: {model_label} ({model_name})")
    print(f"Task: {task_name}")
    print(f"Batch size: {batch_size}")
    print(f"max_gen_toks: {max_gen_toks}")
    print(f"Limit: {limit}")
    print(f"Output dir: {model_dir}")
    print(f"Copy samples to root: {copy_samples_to_root}")

    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    else:
        print("GPU: NOT AVAILABLE ⚠️")

    print("=" * 80 + "\n")

    model_args = [
        f"pretrained={model_name}",
        "dtype=float16",
        "trust_remote_code=True",
    ]

    gen_kwargs = f"max_gen_toks={max_gen_toks},max_new_tokens={max_gen_toks}"

    cmd = [
        "lm-eval", "run",
        "--model", "hf",
        "--model_args", ",".join(model_args),
        "--tasks", task_name,
        "--device", "cuda:0",
        "--batch_size", str(batch_size),
        "--output_path", str(model_dir),
        "--gen_kwargs", gen_kwargs,
        "--log_samples",
    ]

    if chat:
        cmd.append("--apply_chat_template")

    if limit is not None:
        cmd += ["--limit", str(limit)]

    print("[COMMAND]")
    print(" ".join(cmd))
    print("\n[INFO] Launching lm-eval...\n")

    start = time.time()

    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )

    first_output = True

    for line in process.stdout:
        if first_output:
            print("[INFO] lm-eval started producing output...\n")
            first_output = False
        print(line, end="")

    process.wait()
    elapsed = time.time() - start

    if process.returncode != 0:
        print("\n[ERROR] lm-eval failed ❌")
        raise subprocess.CalledProcessError(process.returncode, cmd)

    print("\n[POST] Inspecting output files...")

    json_files = sorted(model_dir.rglob("*.json"))

    if not json_files:
        print("[WARN] No JSON files found in output directory!")
        return model_dir

    print("[DEBUG] JSON files found:")
    for f in json_files:
        print(f" - {f}")

    result_file = None
    sample_files = []

    for f in json_files:
        if "samples" in f.name.lower():
            sample_files.append(f)
            continue

        try:
            data = json.loads(f.read_text(encoding="utf-8"))
            if isinstance(data, dict) and "results" in data and task_name in data["results"]:
                result_file = f
        except Exception:
            pass

    clean_results_path = None

    if result_file is None:
        print("[WARN] No aggregate results file with 'results' found.")
    else:
        data = json.loads(result_file.read_text(encoding="utf-8"))
        task_results = data["results"][task_name]

        clean_results_path = model_dir / "results_clean.json"
        with open(clean_results_path, "w", encoding="utf-8") as f:
            json.dump(task_results, f, indent=2)

        print(f"[SAVED] Clean results -> {clean_results_path}")

    copied_sample_paths = []

    if sample_files:
        print("[INFO] Sample files found:")
        for sf in sample_files:
            print(f" - {sf}")

        if copy_samples_to_root:
            for i, sf in enumerate(sample_files):
                target_name = f"samples_{i}_{sf.name}" if (model_dir / sf.name).exists() else sf.name
                target_path = model_dir / target_name

                if sf.resolve() != target_path.resolve():
                    shutil.copy2(sf, target_path)
                    copied_sample_paths.append(target_path)
                    print(f"[COPIED] {sf} -> {target_path}")
    else:
        print("[WARN] No sample files found, even though --log_samples was enabled.")

    print("\n" + "=" * 80)
    print(f"[DONE] {model_label} | {task_name}")
    print(f"Time: {elapsed / 60:.2f} minutes")
    if clean_results_path is not None:
        print(f"Final aggregate file: {clean_results_path}")
    if copied_sample_paths:
        print("Copied sample files:")
        for p in copied_sample_paths:
            print(f" - {p}")
    print("=" * 80 + "\n")

    return model_dir

In [ ]:
test_out = run_lm_eval(
    model_label="Qwen2-0.5B",
    model_name="Qwen/Qwen2-0.5B",
    task_name="squadv2",
    chat=False,
    batch_size=8,
    limit=200,
    max_gen_toks=32,
)

In [14]:
LIMIT = 100
all_out_dirs = []

for model in MODELS:
    for task in TASKS:
        try:
            out_dir = run_lm_eval(
                model_label=model["label"],
                model_name=model["hf"],
                task_name=task,
                chat=model["chat"],
                batch_size=32,
                limit=LIMIT,
            )
            all_out_dirs.append((model["label"], task, out_dir))
        except subprocess.CalledProcessError as e:
            print(f"[FAILED] {model['label']} | {task}")
            print(e)


[START] Evaluation
Model: Qwen2-1.5B-Instruct (Qwen/Qwen2-1.5B-Instruct)
Task: squadv2
Batch size: 32
max_gen_toks: 32
Limit: 100
Output dir: /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/eval/baseline_evaluations/Qwen2-1.5B-Instruct/squadv2
Copy samples to root: True
GPU: NVIDIA A100-SXM4-40GB

[COMMAND]
lm-eval run --model hf --model_args pretrained=Qwen/Qwen2-1.5B-Instruct,dtype=float16,trust_remote_code=True --tasks squadv2 --device cuda:0 --batch_size 32 --output_path /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/eval/baseline_evaluations/Qwen2-1.5B-Instruct/squadv2 --gen_kwargs max_gen_toks=32,max_new_tokens=32 --log_samples --apply_chat_template --limit 100

[INFO] Launching lm-eval...

[INFO] lm-eval started producing output...

2026-03-22:15:55:41 WARNING  [config.evaluate_config:281] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-03-22:15:55:41 INFO     [config.evaluate_config:301] Using default fewshot_as_m

In [ ]:
for model_label, task, out_dir in all_out_dirs:
    print(f"\n=== {model_label} | {task} ===")
    for p in sorted(out_dir.rglob("*")):
        if p.is_file():
            print(p)

In [ ]:
import json
import pandas as pd

def find_results_json(out_dir: Path):
    candidates = [p for p in out_dir.rglob("*.json") if "samples" not in p.name.lower()]
    for p in candidates:
        try:
            obj = json.loads(p.read_text())
            if isinstance(obj, dict) and "results" in obj:
                return p
        except Exception:
            pass
    return None

rows = []

for model_label, task, out_dir in all_out_dirs:
    result_file = find_results_json(out_dir)

    if result_file is None:
        print(f"[WARN] No result json found for {model_label} | {task}")
        continue

    data = json.loads(result_file.read_text())
    results = data.get("results", {})

    row = {"model": model_label, "task": task}

    if task == "squadv2":
        row["EM"] = results.get("exact,none")
        row["F1"] = results.get("f1,none")
    elif task == "coqa":
        row["EM"] = results.get("em,none") or results.get("exact,none")
        row["F1"] = results.get("f1,none")
    elif task == "hotpotqa":
        row["EM"] = (
            results.get("em,none")
            or results.get("exact_match,none")
            or results.get("exact,none")
        )
        row["F1"] = results.get("f1,none")

    rows.append(row)

df = pd.DataFrame(rows)
df

In [ ]:
summary = []

for model in [m["label"] for m in MODELS]:
    row = {"model": model}

    sq = df[(df["model"] == model) & (df["task"] == "squadv2")]
    cq = df[(df["model"] == model) & (df["task"] == "coqa")]
    hp = df[(df["model"] == model) & (df["task"] == "hotpotqa")]

    row["SQuADv2 EM"] = None if sq.empty else sq.iloc[0]["EM"]
    row["SQuADv2 F1"] = None if sq.empty else sq.iloc[0]["F1"]
    row["CoQA EM"] = None if cq.empty else cq.iloc[0]["EM"]
    row["CoQA F1"] = None if cq.empty else cq.iloc[0]["F1"]
    row["HotpotQA EM"] = None if hp.empty else hp.iloc[0]["EM"]
    row["HotpotQA F1"] = None if hp.empty else hp.iloc[0]["F1"]

    summary.append(row)

summary_df = pd.DataFrame(summary)

for col in summary_df.columns[1:]:
    summary_df[col] = summary_df[col].map(lambda x: round(float(x), 2) if pd.notna(x) else x)

summary_df

In [ ]:
summary_csv_path = BASELINE_DIR / "baseline_summary.csv"
summary_df.to_csv(summary_csv_path, index=False)

print(f"[SAVED] Summary CSV: {summary_csv_path}")

In [ ]:
#From here on cells for eval with hotpotQA

In [ ]:
import os, json, torch, re, string
from tqdm import tqdm
from collections import Counter
from transformers import AutoTokenizer, AutoModelForCausalLM

def build_context(example):
    paragraphs = []
    for paragraph_title, sentences in example["context"]:
        paragraphs.append(f"[{paragraph_title}] " + " ".join(sentences))
    return "\n".join(paragraphs)

def get_max_input_len(tokenizer, model):
    tmax = getattr(tokenizer, "model_max_length", None)
    if tmax and tmax < 10**6:
        return int(tmax)
    mmax = getattr(model.config, "max_position_embeddings", None)
    if mmax:
        return int(mmax)
    return 2048

In [ ]:
import os, json, torch, gc
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

@torch.inference_mode()
def generate_hotpot_predictions_baseline_batched(
    model_name: str,
    hotpot_path: str,
    out_path: str,
    limit: int | None = None,
    max_new_tokens: int = 32,
    batch_size: int = 8,
    overwrite: bool = False,
    verbose: bool = True,
    apply_chat_template: bool = False,
):
    if (not overwrite) and os.path.exists(out_path):
        if verbose:
            print(f"[cache] Using existing predictions: {out_path}")
        return out_path

    if verbose:
        print("=" * 80)
        print("[START] Loading baseline model")
        print(f"Model: {model_name}")
        print(f"Hotpot path: {hotpot_path}")
        print(f"Output path: {out_path}")
        print(f"Batch size: {batch_size}")
        print(f"Max new tokens: {max_new_tokens}")
        print(f"Apply chat template: {apply_chat_template}")
        if torch.cuda.is_available():
            print(f"GPU: {torch.cuda.get_device_name(0)}")
        else:
            print("GPU: NOT AVAILABLE")
        print("=" * 80)

    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        trust_remote_code=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        trust_remote_code=True,
    ).cuda().eval()

    tokenizer.padding_side = "left"

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    pad_id = tokenizer.pad_token_id
    eos_id = tokenizer.eos_token_id

    vocab = int(model.config.vocab_size)
    if pad_id is None or pad_id >= vocab:
        raise ValueError(f"Bad pad_token_id={pad_id} for vocab_size={vocab}")

    max_inp = get_max_input_len(tokenizer, model)
    max_prompt_len = max(1, max_inp - max_new_tokens)

    with open(hotpot_path, "r", encoding="utf-8") as f:
        examples = json.load(f)

    if limit is None:
        limit = len(examples)
    limit = min(limit, len(examples))

    predictions = {"answer": {}, "sp": {}}

    for start in tqdm(range(0, limit, batch_size), desc="Hotpot baseline batched gen"):
        batch = examples[start:start + batch_size]
        batch_ids, batch_prompts = [], []

        for ex in batch:
            qid = ex["_id"]
            question = ex["question"]
            context_text = build_context(ex)

            user_prompt = (
                "Answer the question using the context. "
                "Reply only with a short answer.\n\n"
                f"Context:\n{context_text}\n\n"
                f"Question: {question}\n"
                "Answer:"
            )

            if apply_chat_template:
                messages = [{"role": "user", "content": user_prompt}]
                prompt = tokenizer.apply_chat_template(
                    messages,
                    tokenize=False,
                    add_generation_prompt=True,
                )
            else:
                prompt = user_prompt

            batch_ids.append(qid)
            batch_prompts.append(prompt)

        inputs = tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_prompt_len,
        ).to(model.device)

        max_id = int(inputs["input_ids"].max().item())
        min_id = int(inputs["input_ids"].min().item())
        if max_id >= vocab or min_id < 0:
            raise ValueError(f"Token id out of range: min={min_id}, max={max_id}, vocab_size={vocab}")

        gen = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
            eos_token_id=eos_id,
            pad_token_id=pad_id,
            use_cache=True,
        )

        input_lengths = inputs["attention_mask"].sum(dim=1).tolist()

        for i, qid in enumerate(batch_ids):
            continuation_ids = gen[i][input_lengths[i]:]
            completion = tokenizer.decode(continuation_ids, skip_special_tokens=True).strip()
            predictions["answer"][qid] = completion.splitlines()[0].strip() if completion else ""
            predictions["sp"][qid] = []

    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(predictions, f)

    del model
    torch.cuda.empty_cache()
    gc.collect()

    if verbose:
        print(f"[OK] Saved predictions: {out_path}")

    return out_path

In [ ]:
def normalize_answer(s):
    def remove_articles(text):
        return re.sub(r"\b(a|an|the)\b", " ", text)
    def white_space_fix(text):
        return " ".join(text.split())
    def remove_punc(text):
        exclude = set(string.punctuation)
        return "".join(ch for ch in text if ch not in exclude)
    def lower(text):
        return text.lower()
    return white_space_fix(remove_articles(remove_punc(lower(s))))

def f1_score(prediction, ground_truth):
    normalized_prediction = normalize_answer(prediction)
    normalized_ground_truth = normalize_answer(ground_truth)

    ZERO_METRIC = (0, 0, 0)

    if normalized_prediction in ["yes", "no", "noanswer"] and normalized_prediction != normalized_ground_truth:
        return ZERO_METRIC
    if normalized_ground_truth in ["yes", "no", "noanswer"] and normalized_prediction != normalized_ground_truth:
        return ZERO_METRIC

    prediction_tokens = normalized_prediction.split()
    ground_truth_tokens = normalized_ground_truth.split()
    common = Counter(prediction_tokens) & Counter(ground_truth_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return ZERO_METRIC

    precision = 1.0 * num_same / len(prediction_tokens)
    recall = 1.0 * num_same / len(ground_truth_tokens)
    f1 = (2 * precision * recall) / (precision + recall)
    return f1, precision, recall

def exact_match_score(prediction, ground_truth):
    return normalize_answer(prediction) == normalize_answer(ground_truth)

def update_answer(metrics, prediction, gold):
    em = exact_match_score(prediction, gold)
    f1, prec, recall = f1_score(prediction, gold)
    metrics["em"] += float(em)
    metrics["f1"] += f1
    metrics["prec"] += prec
    metrics["recall"] += recall
    return em, prec, recall

def update_sp(metrics, prediction, gold):
    cur_sp_pred = set(map(tuple, prediction))
    gold_sp_pred = set(map(tuple, gold))

    tp = sum(1 for e in cur_sp_pred if e in gold_sp_pred)
    fp = sum(1 for e in cur_sp_pred if e not in gold_sp_pred)
    fn = sum(1 for e in gold_sp_pred if e not in cur_sp_pred)

    prec = 1.0 * tp / (tp + fp) if tp + fp > 0 else 0.0
    recall = 1.0 * tp / (tp + fn) if tp + fn > 0 else 0.0
    f1 = 2 * prec * recall / (prec + recall) if prec + recall > 0 else 0.0
    em = 1.0 if fp + fn == 0 else 0.0

    metrics["sp_em"] += em
    metrics["sp_f1"] += f1
    metrics["sp_prec"] += prec
    metrics["sp_recall"] += recall
    return em, prec, recall

def hotpot_eval(prediction_file, gold_file):
    with open(prediction_file, "r", encoding="utf-8") as f:
        prediction = json.load(f)
    with open(gold_file, "r", encoding="utf-8") as f:
        gold = json.load(f)

    metrics = {
        "em": 0, "f1": 0, "prec": 0, "recall": 0,
        "sp_em": 0, "sp_f1": 0, "sp_prec": 0, "sp_recall": 0,
        "joint_em": 0, "joint_f1": 0, "joint_prec": 0, "joint_recall": 0,
    }

    for dp in gold:
        cur_id = dp["_id"]
        can_eval_joint = True

        if cur_id not in prediction["answer"]:
            print(f"missing answer {cur_id}")
            can_eval_joint = False
        else:
            em, prec, recall = update_answer(metrics, prediction["answer"][cur_id], dp["answer"])

        if cur_id not in prediction["sp"]:
            print(f"missing sp fact {cur_id}")
            can_eval_joint = False
        else:
            sp_em, sp_prec, sp_recall = update_sp(metrics, prediction["sp"][cur_id], dp["supporting_facts"])

        if can_eval_joint:
            joint_prec = prec * sp_prec
            joint_recall = recall * sp_recall
            joint_f1 = (2 * joint_prec * joint_recall / (joint_prec + joint_recall)) if (joint_prec + joint_recall) > 0 else 0.0
            joint_em = em * sp_em

            metrics["joint_em"] += joint_em
            metrics["joint_f1"] += joint_f1
            metrics["joint_prec"] += joint_prec
            metrics["joint_recall"] += joint_recall

    N = len(gold)
    for k in list(metrics.keys()):
        metrics[k] /= N

    return metrics

In [ ]:
import os
import json

hotpot_path = "/content/drive/MyDrive/TUM/Pratikum/code/sliced_rag/data/hotpot_dev_distractor_v1.json"

limit = None
max_new_tokens = 32
batch_size = 32

all_results = []

for model in MODELS:
    model_label = model["label"]
    model_hf = model["hf"]
    use_chat = model["chat"]

    print("\n" + "="*80)
    print(f"[MODEL] {model_label}")
    print("="*80)

    # ===== CLEAN DIRECTORY STRUCTURE =====
    model_dir = BASELINE_DIR / model_label / "hotpotqa"
    os.makedirs(model_dir, exist_ok=True)

    pred_path = model_dir / "predictions.json"
    results_path = model_dir / "results.json"

    # ===== GENERATE PREDICTIONS =====
    pred_path = generate_hotpot_predictions_baseline_batched(
        model_name=model_hf,
        hotpot_path=hotpot_path,
        out_path=str(pred_path),
        limit=limit,
        max_new_tokens=max_new_tokens,
        batch_size=batch_size,
        overwrite=False,
        verbose=True,
        apply_chat_template=use_chat,
    )

    # ===== EVALUATE =====
    metrics = hotpot_eval(str(pred_path), hotpot_path)

    # ===== SAVE CLEAN RESULTS (ONLY METRICS) =====
    with open(results_path, "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2)

    print(f"[SAVED] {results_path}")

    # collect for summary
    all_results.append({
        "model": model_label,
        **metrics
    })

print("\n" + "="*80)
print("[FINAL RESULTS]")
for r in all_results:
    print(r)

Hotpot baseline batched gen: 100%|██████████| 232/232 [12:29<00:00,  3.23s/it]


[OK] Saved predictions: /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/eval/baseline_evaluations/Qwen2-1.5B-Instruct/hotpotqa/predictions.json
[SAVED] /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/eval/baseline_evaluations/Qwen2-1.5B-Instruct/hotpotqa/results.json

[MODEL] OPT-125M
[START] Loading baseline model
Model: facebook/opt-125m
Hotpot path: /content/drive/MyDrive/TUM/Pratikum/code/sliced_rag/data/hotpot_dev_distractor_v1.json
Output path: /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/eval/baseline_evaluations/OPT-125M/hotpotqa/predictions.json
Batch size: 32
Max new tokens: 32
Apply chat template: False
GPU: NVIDIA A100-SXM4-40GB


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Hotpot baseline batched gen: 100%|██████████| 232/232 [02:05<00:00,  1.85it/s]


[OK] Saved predictions: /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/eval/baseline_evaluations/OPT-125M/hotpotqa/predictions.json
[SAVED] /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/eval/baseline_evaluations/OPT-125M/hotpotqa/results.json

[MODEL] OPT-1.3B
[START] Loading baseline model
Model: facebook/opt-1.3b
Hotpot path: /content/drive/MyDrive/TUM/Pratikum/code/sliced_rag/data/hotpot_dev_distractor_v1.json
Output path: /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/eval/baseline_evaluations/OPT-1.3B/hotpotqa/predictions.json
Batch size: 32
Max new tokens: 32
Apply chat template: False
GPU: NVIDIA A100-SXM4-40GB


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Hotpot baseline batched gen: 100%|██████████| 232/232 [09:36<00:00,  2.49s/it]


[OK] Saved predictions: /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/eval/baseline_evaluations/OPT-1.3B/hotpotqa/predictions.json
[SAVED] /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/eval/baseline_evaluations/OPT-1.3B/hotpotqa/results.json

[FINAL RESULTS]
{'model': 'Qwen2-0.5B', 'em': 0.004186360567184335, 'f1': 0.020226492048677592, 'prec': 0.014636265383469557, 'recall': 0.11383014202393044, 'sp_em': 0.0, 'sp_f1': 0.0, 'sp_prec': 0.0, 'sp_recall': 0.0, 'joint_em': 0.0, 'joint_f1': 0.0, 'joint_prec': 0.0, 'joint_recall': 0.0}
{'model': 'Qwen2-1.5B-Instruct', 'em': 0.009993247805536799, 'f1': 0.02739172412942758, 'prec': 0.022394258598438587, 'recall': 0.11860673275258056, 'sp_em': 0.0, 'sp_f1': 0.0, 'sp_prec': 0.0, 'sp_recall': 0.0, 'joint_em': 0.0, 'joint_f1': 0.0, 'joint_prec': 0.0, 'joint_recall': 0.0}
{'model': 'OPT-125M', 'em': 0.0004051316677920324, 'f1': 0.01586003252956028, 'prec': 0.009569170436453665, 'recall': 0.12253341114583557, 'sp_em': 0.0, 'sp_f1